# 第3章 数据包络分析的经典模型

> 由书稿（新 API）自动重建（分省份）：直接取自 .md 的 分省份 代码块。

In [ ]:
# 导入中国省份数据（本例为省级），筛选 2015 年后并重置索引
dmuname = '省份'  # DMU 名称为省份
data = (
    pd.read_excel(data_path("china data.xlsx"))
    .query('year>2015')
    .reset_index(drop=True)
)
region = data[["省份", "group"]]  # 地区分组（东/中/西）

# 省级 DDF：复用 contemporaneous() 骨架，方向 g=(x,0)
def ddf_vrs_factory(d):
    return DEA.ddf(d, sent="K L E=Y", gy=[0], gx=[1, 1, 1], rts=RTS_VRS)

DDF_VRS = (contemporaneous(ddf_vrs_factory, data, dmuname)
           .assign(TECH_VRS=lambda d: 1 - d['rho'])   # 投入方向：te = 1 - ρ
           .drop(columns=['optimization_status', 'rho', 'objective_value']))
DDF_VRS3 = DDF_VRS.pivot_table(values=['TECH_VRS'], index=dmuname, columns='year')


In [ ]:
# 省级：各年份平均技术效率
ddf_mean_year = DDF_VRS3['TECH_VRS'].mean()
xtick2 = tuple(ddf_mean_year.index)

import matplotlib.pyplot as plt
import numpy as np
from matplotlib import rcParams
rcParams['font.family'] = 'SimHei'

fig, ax = plt.subplots(figsize=(8,4))
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.plot(np.arange(len(ddf_mean_year)), ddf_mean_year.values,
        color='darkblue', alpha=0.8, linestyle="dashed", marker="s", label='技术效率趋势')
ax.set_ylabel('技术效率')
ax.set_xticks(np.arange(len(ddf_mean_year)))
ax.set_xticklabels(xtick2, rotation=80)
plt.legend()
plt.savefig("chap3分年份技术效率-省.png", dpi=900, bbox_inches='tight')
plt.show()


In [ ]:
# 省级：各省份平均技术效率并排序
ddf_mean_cou = DDF_VRS3['TECH_VRS'].mean(axis=1).sort_values()
xtick2 = tuple(ddf_mean_cou.index)

import matplotlib.pyplot as plt
import numpy as np
from matplotlib import rcParams
rcParams['font.family'] = 'SimHei'

fig, ax = plt.subplots(figsize=(12,6))
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
n_groups = 30
index = np.arange(n_groups)
ax.bar(index, ddf_mean_cou, 0.5/2, color='olivedrab', alpha=0.8, edgecolor='white', label='技术效率')
ax.set_ylabel('技术效率', fontsize=12)
ax.set_xticks(index)
ax.set_xticklabels(xtick2, rotation=80)
plt.legend(loc='upper left')
plt.savefig("chap3分省份技术效率.png", dpi=900, bbox_inches='tight')
plt.show()


In [ ]:
# 按地区分组分析技术效率（省级数据）
# 合并地区分组信息并计算各区域年均效率
DDF_VRS4 = pd.merge(DDF_VRS3['TECH_VRS'].reset_index(),
                   region.drop_duplicates(),
                   on=['省份'], how="left")
DDF_VRS5 = DDF_VRS4.drop(columns=['省份']).groupby('group').mean()

# 准备绘图数据
xtick2 = tuple(DDF_VRS5.loc["W"].index)  # 使用西部地区的年份作为x轴标签

# 创建图形
fig, ax = plt.subplots(figsize=(8,5))
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

# 绘制三条趋势线（东部/中部/西部）
ax.plot(np.arange(4), DDF_VRS5.loc['E'],  # 东部
        color='darkblue', alpha=0.8,
        linestyle="dashed", marker="s",
        label='东部')

ax.plot(np.arange(4), DDF_VRS5.loc['C'],  # 中部
        color='olivedrab', alpha=0.8,
        linestyle="dashed", marker="^",
        label='中部')

ax.plot(np.arange(4), DDF_VRS5.loc['W'],  # 西部
        color='orange', alpha=0.8,
        linestyle="dashed", marker="+",
        label='西部')

# 设置图表格式
ax.set_ylabel('技术效率', fontsize=12)
ax.set_xticks(np.arange(4))
ax.set_xticklabels(xtick2, rotation=80)
plt.legend(loc='best')

# 保存图表
plt.savefig("chap3分年份地区技术效率-省.png",
           dpi=900,
           bbox_inches='tight')
plt.show()
